# 学習したTTSモデルを使って音声を合成する
このファイルを`espnet/egs2/○○/tts1/`フォルダに置いて、上のブロックから順番に実行する

In [4]:
from espnet2.bin.tts_inference import Text2Speech
from espnet2.utils.types import str_or_none

モデルと設定ファイルのパスを記述する<br>
妙な処理をしていなければどちらのファイルも`exp/tts_train_○○/`にあるはず

In [5]:
# /home/{ユーザー名}/espnet2/egs2/{レシピ名}/tts1
%cd /home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets_deltaK2/tts1

import os

# 音声をwavファイルに出力するかどうか
save_wavfile = True
# 音声ファイルの保存にscipyかpysoundfile(sf)どちらを使うか
# save_method = "scipy"
save_method = "sf"
# モデルのパス
model = "/home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets_deltaK2/tts1/exp/22.05k/tts_train_jets_22.05k_raw_phn_jaconv_pyopenjtalk_prosody/1000epoch.pth"
# 学習時の設定ファイルのパス (指定しなければモデルのパスから勝手に探す)
config = "/home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets_deltaK2/tts1/exp/22.05k/tts_train_jets_22.05k_raw_phn_jaconv_pyopenjtalk_prosody/config.yaml"
if config == None or config == '':
    config = os.path.join(os.path.dirname(model), "config.yaml")


# colabのサンプルに書いてあったもの
# @title Choose Japanese model { run: "auto" }
# lang = 'Japanese'
# tag = 'kan-bayashi/jsut_full_band_vits_prosody' #@param ["kan-bayashi/jsut_tacotron2", "kan-bayashi/jsut_transformer", "kan-bayashi/jsut_fastspeech", "kan-bayashi/jsut_fastspeech2", "kan-bayashi/jsut_conformer_fastspeech2", "kan-bayashi/jsut_conformer_fastspeech2_accent", "kan-bayashi/jsut_conformer_fastspeech2_accent_with_pause", "kan-bayashi/jsut_vits_accent_with_pause", "kan-bayashi/jsut_full_band_vits_accent_with_pause", "kan-bayashi/jsut_tacotron2_prosody", "kan-bayashi/jsut_transformer_prosody", "kan-bayashi/jsut_conformer_fastspeech2_tacotron2_prosody", "kan-bayashi/jsut_vits_prosody", "kan-bayashi/jsut_full_band_vits_prosody", "kan-bayashi/jvs_jvs010_vits_prosody", "kan-bayashi/tsukuyomi_full_band_vits_prosody"] {type:"string"}
# vocoder_tag = 'none' #@param ["none", "parallel_wavegan/jsut_parallel_wavegan.v1", "parallel_wavegan/jsut_multi_band_melgan.v2", "parallel_wavegan/jsut_style_melgan.v1", "parallel_wavegan/jsut_hifigan.v1"] {type:"string"}

/home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets_deltaK2/tts1


In [3]:
# d0_control_alphaの値は後のセルでループ処理するため、ここでは設定しない
# text2speechオブジェクトは後のセルで各d0_control_alpha値ごとに作成します

In [6]:
# ─── 必要ライブラリ ─────────────────────────────────────────
from IPython.display import Audio, display
import torch, os, re
from pathlib import Path
import numpy as np

# 保存方法の選択（既存フラグをそのまま利用）
if save_method == "sf":
    import soundfile as sf
elif save_method == "scipy":
    from scipy.io.wavfile import write

# ─── ① ここに好きな文を並べる ───────────────────────────────
texts = [
    "水をマレーシアから買わなくてはならないのです。",
    "木曜日、停戦会談は、何の進展もないまま終了しました。",
    "上院議員は私がデータをゆがめたと告発した。",
    "１週間して、そのニュースは本当になった。",
    "血圧は、健康のパロメーターとして重要である。",
    "週に四回、フランスの授業があります。",
    "許可書がなければここへは入れない。",
    "大声で泣きながら、女の子は母親を探していた。",
    "無罪の人々は、もちろん放免された。",
    "末期試験に備えて、本当に気合いを入れて勉強しなきゃ。",
    "木曽川は、しばしば日本のライン川と呼ばれている。",
    "溺れかかっていた乗客は、すべて救助された。",
    "中心部にあるので、商店や、オフィスに行くのに便利です。",
    "残酷ということは、彼の性質にはないことだ。",
    "庭園の周りに、ぐるりと高いへいが立っている。",
    "出生率と死亡率は、ほぼ等しかった。",
    "絶対にそのスイッチに触ってはいけない。",
    "部屋を出るときには、明かりを消しなさい。",
    "シェークスピアの作品は、とてもむずかしくて読めない。",
    "むかし、むかし、１人の老人が住んでいた。",
    "疲労やら、飢えやらで、彼は目眩を感じた。",
    "むしろロン毛のほうが禿げやすいって聞いたぞ。",
    "むこうに立っている女の子は、メアリーです。",
    "システィナ礼拝堂は、１４７３年に、バティカン宮殿内に建立された、壮大な礼拝堂です。",
    "テニスにもあるけど、４大大会って何。",
    "策略は衣服を必要とするが、真実は裸であることを好む。",
    "濃いコーヒーより、うすいコーヒーの方が好きです。",
    "外交官には、様々な特権が与えられている。",
    "刺し傷はとても深く、感染の恐れがないか検査する必要がある。",
    "外人が日本食に慣れることはむずかしい。",
    "語の古い意味が、現在の基本的な意味であるとは限らない。",
    "針金は、電気を伝えるのにもちいられる。",
    "来月の歌舞伎座の出し物はなんですか。",
    "圧縮したファイルを添付で送ってください。",
    "容姿端麗、頭脳明晰、運動神経抜群、家は金持ちで、ついでに学生会の副会長をしてたりもする、いわゆる、パーフェクトな奴だ。",
    "入院患者は、医者に麻酔を注射されて、すぐに眠りに落ちた。",
    "傍目八目という言葉があるように一度協会から離れて、日本サッカーをみて頂きたい。",
    "行儀の悪さは、彼の良識を疑わせるものだ。",
    "部長の都合が悪くなってしまったので、飲み会の日程は仕切り直しだね。",
    "蛇をみたとき、彼は悲鳴をあげた。",
    "紫外線は、皮膚癌を引き起こすことがある。",
    "気がつくと、逃げ場はどこにもなかった。",
    "三番目の、そしてもっとも重要な考えは、再入ということである。",
    "希望や夢の思いは、絶対に見つからない。",
    "前方にある、あのサインが読めますか。",
    "気まずい沈黙の後、ビルは彼女の手を取って、上の階へ引っ張って行った。",
    "鉱山労働者が、賃上げを要求してストに突入した。",
    "気の弱い男が、美女を得たためしがない。",
    "気前の良いその歯科医は、およそ２０億円を、慈善事業に寄付した。",
    "気に入ろうが入るまいが、君は行かねばならない。",  
]

# ─── ② d0_control_alphaの値リストを生成 ───────────────────────
# 1.0から2.0まで0.1刻み
d0_alpha_values_1 = np.arange(1.0, 2.1, 0.1)
# 2.0から6.0まで1刻み (2.0は上と重複するので3.0から)
d0_alpha_values_2 = np.arange(3.0, 6.1, 1.0)
# 結合
d0_alpha_values = np.concatenate([d0_alpha_values_1, d0_alpha_values_2])

print(f"生成する d0_control_alpha の値: {d0_alpha_values}")
print(f"合計 {len(d0_alpha_values)} パターン × {len(texts)} 文 = {len(d0_alpha_values) * len(texts)} 件の音声を生成します\n")

# ─── ③ d0_control_alpha ごとにループ処理 ─────────────────────
for d0_alpha in d0_alpha_values:
    d0_alpha = round(float(d0_alpha), 2)  # 丸め誤差対策
    print(f"{'='*60}")
    print(f"d0_control_alpha = {d0_alpha} の音声を生成中...")
    print(f"{'='*60}")
    
    # Text2Speechオブジェクトを作成
    text2speech = Text2Speech.from_pretrained(
        model_file=model,
        train_config=config,
        device="cpu",
        threshold=0.5,
        minlenratio=0.0,
        maxlenratio=10.0,
        use_att_constraint=False,
        backward_window=1,
        forward_window=3,
        speed_control_alpha=1.0,
        noise_scale=0.333,
        noise_scale_dur=0.333,
        d0_control_alpha=d0_alpha,
    )
    
    # 出力フォルダの用意
    out_dir = Path(f"inference_d0_control_{d0_alpha:.2f}")
    out_dir.mkdir(exist_ok=True)
    
    # テキストごとにループで推論・保存
    for idx, text in enumerate(texts, start=1):
        # 推論
        with torch.no_grad():
            wav_tensor = text2speech(text)["wav"]
        audio_array = wav_tensor.view(-1).cpu().numpy()
        samplerate = text2speech.fs

        # ファイル名生成（安全文字だけ残す）
        safe = re.sub(r"[^\w]", "_", text[:10])
        filename = f"{idx:02d}_{safe}.wav"
        filepath = out_dir / filename

        # 保存
        if save_wavfile:
            if save_method == "sf":
                sf.write(filepath, audio_array, samplerate)
            elif save_method == "scipy":
                write(filepath, samplerate, audio_array)

    print(f"✅ {len(texts)} 件の音声を {out_dir} に保存しました\n")

print(f"{'='*60}")
print(f"すべての処理が完了しました！")
print(f"合計 {len(d0_alpha_values) * len(texts)} 件の音声を生成しました。")
print(f"{'='*60}")


生成する d0_control_alpha の値: [1.  1.1 1.2 1.3 1.4 1.5 1.6 1.7 1.8 1.9 2.  3.  4.  5.  6. ]
合計 15 パターン × 50 文 = 750 件の音声を生成します

d0_control_alpha = 1.0 の音声を生成中...
Debug - Filtered kwargs: {'model_file': '/home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets_deltaK2/tts1/exp/22.05k/tts_train_jets_22.05k_raw_phn_jaconv_pyopenjtalk_prosody/1000epoch.pth', 'train_config': '/home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets_deltaK2/tts1/exp/22.05k/tts_train_jets_22.05k_raw_phn_jaconv_pyopenjtalk_prosody/config.yaml', 'device': 'cpu', 'threshold': 0.5, 'minlenratio': 0.0, 'maxlenratio': 10.0, 'use_att_constraint': False, 'backward_window': 1, 'forward_window': 3, 'speed_control_alpha': 1.0, 'noise_scale': 0.333, 'noise_scale_dur': 0.333, 'd0_control_alpha': 1.0}
Debug - Argument types:
  use_teacher_forcing: <class 'bool'> = False
  d0_control_alpha: <class 'float'> = 1.0
  speed_control_alpha: <class 'float'> = 1.0
  noise_scale: <class 'float'> = 0.333
  always_fix_seed: <class 'bool'>

/home/uesugi/miniconda3/envs/espnet_custom/lib/python3.9/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/home/uesugi/miniconda3/envs/espnet_custom/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 50 件の音声を inference_d0_control_1.00 に保存しました

d0_control_alpha = 1.1 の音声を生成中...
Debug - Filtered kwargs: {'model_file': '/home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets_deltaK2/tts1/exp/22.05k/tts_train_jets_22.05k_raw_phn_jaconv_pyopenjtalk_prosody/1000epoch.pth', 'train_config': '/home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets_deltaK2/tts1/exp/22.05k/tts_train_jets_22.05k_raw_phn_jaconv_pyopenjtalk_prosody/config.yaml', 'device': 'cpu', 'threshold': 0.5, 'minlenratio': 0.0, 'maxlenratio': 10.0, 'use_att_constraint': False, 'backward_window': 1, 'forward_window': 3, 'speed_control_alpha': 1.0, 'noise_scale': 0.333, 'noise_scale_dur': 0.333, 'd0_control_alpha': 1.1}
Debug - Argument types:
  use_teacher_forcing: <class 'bool'> = False
  d0_control_alpha: <class 'float'> = 1.1
  speed_control_alpha: <class 'float'> = 1.0
  noise_scale: <class 'float'> = 0.333
  always_fix_seed: <class 'bool'> = False


/home/uesugi/miniconda3/envs/espnet_custom/lib/python3.9/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


✅ 50 件の音声を inference_d0_control_1.10 に保存しました

d0_control_alpha = 1.2 の音声を生成中...
Debug - Filtered kwargs: {'model_file': '/home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets_deltaK2/tts1/exp/22.05k/tts_train_jets_22.05k_raw_phn_jaconv_pyopenjtalk_prosody/1000epoch.pth', 'train_config': '/home/uesugi/espnet-for-customize-jets/egs2/jsut_d0_jets_deltaK2/tts1/exp/22.05k/tts_train_jets_22.05k_raw_phn_jaconv_pyopenjtalk_prosody/config.yaml', 'device': 'cpu', 'threshold': 0.5, 'minlenratio': 0.0, 'maxlenratio': 10.0, 'use_att_constraint': False, 'backward_window': 1, 'forward_window': 3, 'speed_control_alpha': 1.0, 'noise_scale': 0.333, 'noise_scale_dur': 0.333, 'd0_control_alpha': 1.2}
Debug - Argument types:
  use_teacher_forcing: <class 'bool'> = False
  d0_control_alpha: <class 'float'> = 1.2
  speed_control_alpha: <class 'float'> = 1.0
  noise_scale: <class 'float'> = 0.333
  always_fix_seed: <class 'bool'> = False
✅ 50 件の音声を inference_d0_control_1.20 に保存しました

d0_control_alpha = 1.3 